# Sprint 1 Role 2 — Data Validation

Re-runnable companion to `docs/data_validation.md`. Loads the merged
`opportunity_df` from `src.opportunity_cleaner.build_opportunity_df()` and
calls each summary function in `src.data_validator` so every number in the
notes document can be reproduced from a clean kernel.

Run order: setup → missingness → revenue → date/duration → categorical →
owner identity → owner aggregates → reliable-vs-risky.

## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

from src.opportunity_cleaner import build_opportunity_df
from src.data_validator import (
    Fields,
    apply_revenue_hierarchy,
    build_owner_aggregates,
    summarize_missingness,
    validate_categorical_fields,
    validate_date_duration_fields,
    validate_owner_identity,
    validate_revenue_fields,
)

opportunity_df, schema_comparison, merge_summary, audit_tables = build_opportunity_df()
print(f"opportunity_df: {len(opportunity_df):,} rows, {opportunity_df.shape[1]} columns")

## 1. Missingness summary

One row per validated field. The columns most central to scoring (`status`,
`sales_stage`, `opportunity_owner`, `opportunity_manager`) are 100%
populated; the high null rate on `opportunity_estimated_revenue_base_cad`
is structural (see §2).

In [ ]:
missingness = summarize_missingness(opportunity_df)
missingness

## 2. Revenue field hierarchy

`total_estimated_revenue` is the canonical revenue field; the architecture
spec'd fallback to `opportunity_estimated_revenue_base_cad` covers the 51
opps1-exclusive rows the merge brings in. Service-solution revenue must
**not** be summed onto either.

In [ ]:
validate_revenue_fields(opportunity_df)

In [ ]:
# Apply the hierarchy and inspect what gets routed to fallback
resolved = apply_revenue_hierarchy(opportunity_df)
print("revenue_source value counts:")
print(resolved["revenue_source"].value_counts(dropna=False))
print("\nFallback rows (showing 5):")
(
    opportunity_df.assign(**resolved)
    .loc[resolved["revenue_source"] == "fallback",
         [Fields.OWNER, Fields.STATUS, "authoritative_revenue", "revenue_source"]]
    .head()
)

## 3. Date and duration validation

Critical findings:
1. Delivery window is 100% resolvable via `revenue_start_date` on the Won subset.
2. `n_close_before_created` is large (~35% of rows) — needs CGI clarification.
3. Duration outliers > 60 months persist; clip in `capacity_engine` does not address.

In [ ]:
validate_date_duration_fields(opportunity_df)

In [ ]:
# Investigate close_before_created — distribution by status
created = pd.to_datetime(opportunity_df[Fields.CREATED_ON], errors="coerce")
close = pd.to_datetime(opportunity_df[Fields.CLOSE_DATE], errors="coerce")
delta_days = (close - created).dt.days
anomalies = opportunity_df.assign(_delta_days=delta_days).loc[delta_days < 0]
print(f"Rows with close_date < created_on: {len(anomalies):,}")
print("\nBy status:")
print(anomalies[Fields.STATUS].value_counts())
print("\nDelta distribution (days):")
print(delta_days.loc[delta_days < 0].describe().round(1))

In [ ]:
# Duration outliers
duration = pd.to_numeric(opportunity_df[Fields.DURATION_MONTHS], errors="coerce")
print("Duration distribution (months):")
print(duration.describe().round(1))
print("\nOver 60 months:")
print(duration[duration > 60].describe().round(1))

## 4. Categorical / scoring fields

Confirms `sales_stage` covers exactly the architecture's 7-stage table
(no unmapped values), `probability` stays in [0, 100], and the
`status` vs `status_reason` Won-detection disagreement is a 15-row corner case.

In [ ]:
validate_categorical_fields(opportunity_df)

In [ ]:
# status x status_reason crosstab — justifies architecture's 'use status_reason' rule
pd.crosstab(
    opportunity_df[Fields.STATUS],
    opportunity_df[Fields.STATUS_REASON],
    margins=True,
    margins_name="total",
)

In [ ]:
# sales_stage distribution (used for late_stage_deal_count and stage_weight mapping)
opportunity_df[Fields.SALES_STAGE].value_counts(dropna=False)

In [ ]:
# probability distribution and null rate by status
probability = pd.to_numeric(opportunity_df[Fields.PROBABILITY], errors="coerce")
print("Overall null rate:", probability.isna().mean().round(4))
print("\nNull rate by status:")
print(
    opportunity_df.assign(_prob_null=probability.isna())
    .groupby(Fields.STATUS)["_prob_null"]
    .mean()
    .round(4)
)
print("\nValue distribution:")
print(probability.describe().round(2))

## 5. Owner identity

24 distinct owners, no casing variants, zero owner-equals-manager rows.
Seven owners have fewer than 10 lifetime opportunities — their baselines
should carry a Low reliability flag.

In [ ]:
validate_owner_identity(opportunity_df)

In [ ]:
# Owner deal counts (sorted, top + tail)
owner_counts = opportunity_df[Fields.OWNER].value_counts()
print("Top 5 by lifetime opportunity count:")
print(owner_counts.head())
print("\nBottom 7 (Low reliability candidates):")
print(owner_counts.tail(7))

## 6. Owner aggregates

`build_owner_aggregates(opportunity_df)` produces the seven columns Freya's
dashboard mock expects but Lyken's `director_df` does not (yet) emit.
Lyken's score columns join onto this on `opportunity_owner`.

In [ ]:
owner_aggregates = build_owner_aggregates(opportunity_df)
print(f"{len(owner_aggregates)} owners")
owner_aggregates.sort_values("weighted_pipeline_revenue", ascending=False)

In [ ]:
# Optional local export — overwrites previous run; data/processed/ is gitignored
from pathlib import Path
out_dir = PROJECT_ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
owner_aggregates.to_csv(out_dir / "owner_validation_summary.csv", index=False)
print("Wrote", out_dir / "owner_validation_summary.csv")

## 7. Reliable-vs-risky fields summary

See `docs/data_validation.md` §6 for the human-readable rating
table. Below: a programmatic snapshot of the same colour codes derived
from the missingness table, for inclusion in the dashboard's data-quality
panel later.

In [ ]:
def _rating(row):
    pct = row["pct_null"]
    if row["field"] == Fields.CLOSE_DATE:
        return "red"  # see §3 for ordering anomaly
    if row["field"] == Fields.REVENUE_FALLBACK:
        return "green"  # high null is structural, see §2
    if pct < 1:
        return "green"
    if pct < 15:
        return "yellow"
    return "red"

rating_table = missingness.assign(rating=missingness.apply(_rating, axis=1))[
    ["field", "family", "pct_null", "rating"]
]
rating_table

---

**Use of Generative AI.** Anthropic Claude (Opus 4.7) was used for drafting and editing assistance on this notebook. All numerical findings come from running the validator functions on the team's merged `opportunity_df`. No CGI data was sent to the tool.